In [1]:
import sys
from pathlib import Path
import os

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("CWD:", os.getcwd())
print("PROJECT_ROOT:", PROJECT_ROOT)
print("CONFIGS EXISTS:", (PROJECT_ROOT / "configs").exists())
print("SRC EXISTS:", (PROJECT_ROOT / "src").exists())
print("OUTPUTS EXISTS:", (PROJECT_ROOT / "outputs").exists())

CWD: /home/harielpadillasanchez/Documentos/TT/TT2/notebooks
PROJECT_ROOT: /home/harielpadillasanchez/Documentos/TT/TT2
CONFIGS EXISTS: True
SRC EXISTS: True
OUTPUTS EXISTS: True


In [2]:
import json
import itertools
from datetime import datetime
from pathlib import Path

import pandas as pd

from configs.models import MODELS, DEFAULT_GENERATION_CONFIG
from configs.prompts import ZERO_SHOT_TEMPLATE, build_few_shot_prompt
from src.experiment.schemas import new_record
from src.utils.logging_utils import ExperimentLogger
from src.experiment.runner import ExperimentRunner

/home/harielpadillasanchez/Documentos/TT/TT2/.venv-bloom/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
experiment_id = f"exp_small_batch_v1_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

manifest = {
    "experiment_id": experiment_id,
    "goal": "Small batch de validación operativa para simplificación automática en español",
    "models_to_test": ["mistral", "llama3"],
    "prompt_types": ["zero-shot"],
    "rulesets": ["R0", "R2"],
    "notes": [
        "Validación de estabilidad del pipeline",
        "Comparación inicial entre Mistral y Llama3",
        "BLOOMZ se deja opcional por baja fidelidad semántica observada",
        "Aún sin métricas automáticas; enfoque en calidad preliminar, tiempos y logs"
    ],
    "default_generation_config": DEFAULT_GENERATION_CONFIG,
}

logger = ExperimentLogger(base_dir="outputs/logs")
logger.save_manifest(experiment_id, manifest)

print("Manifest guardado correctamente.")
print("experiment_id:", experiment_id)

Manifest guardado correctamente.
experiment_id: exp_small_batch_v1_20260308_063402


In [4]:
samples = [
    {
        "sample_id": "sb_001",
        "source_text": "Una vez dirimidos estos asuntos, se entra de lleno a algunos aspectos e instrumentos que son propios de las finanzas, pero siempre en la óptica de lo personal y lo familiar."
    },
    {
        "sample_id": "sb_002",
        "source_text": "La implementación de estrategias de planificación financiera permite a las familias optimizar la administración de sus recursos y reducir riesgos asociados al endeudamiento."
    },
    {
        "sample_id": "sb_003",
        "source_text": "El análisis de los instrumentos de ahorro exige considerar variables como liquidez, rendimiento, horizonte temporal y tolerancia al riesgo."
    },
    {
        "sample_id": "sb_004",
        "source_text": "La correcta interpretación de las condiciones contractuales evita malentendidos y permite tomar decisiones informadas respecto a productos financieros."
    },
    {
        "sample_id": "sb_005",
        "source_text": "Los usuarios deben identificar no solo la tasa de interés nominal, sino también las comisiones, penalizaciones y costos adicionales asociados al servicio."
    },
    {
        "sample_id": "sb_006",
        "source_text": "El presupuesto familiar constituye una herramienta fundamental para anticipar gastos, distribuir ingresos y prevenir situaciones de sobreendeudamiento."
    },
    {
        "sample_id": "sb_007",
        "source_text": "Aunque el crédito puede facilitar el acceso inmediato a bienes y servicios, su uso irresponsable puede comprometer la estabilidad económica del hogar."
    },
    {
        "sample_id": "sb_008",
        "source_text": "En diversos contextos educativos, la simplificación textual puede contribuir a mejorar la comprensión lectora sin alterar el contenido esencial del mensaje."
    },
]

samples_df = pd.DataFrame(samples)
samples_df

,sample_id,source_text
0,sb_001,"Una vez dirimidos estos asuntos, se entra de l..."
1,sb_002,La implementación de estrategias de planificac...
2,sb_003,El análisis de los instrumentos de ahorro exig...
3,sb_004,La correcta interpretación de las condiciones ...
4,sb_005,Los usuarios deben identificar no solo la tasa...
5,sb_006,El presupuesto familiar constituye una herrami...
6,sb_007,Aunque el crédito puede facilitar el acceso in...
7,sb_008,"En diversos contextos educativos, la simplific..."


In [5]:
samples_df["n_chars"] = samples_df["source_text"].str.len()
samples_df["n_words"] = samples_df["source_text"].str.split().str.len()

samples_df[["sample_id", "n_chars", "n_words", "source_text"]]

,sample_id,n_chars,n_words,source_text
0,sb_001,173,31,"Una vez dirimidos estos asuntos, se entra de l..."
1,sb_002,173,23,La implementación de estrategias de planificac...
2,sb_003,139,19,El análisis de los instrumentos de ahorro exig...
3,sb_004,151,18,La correcta interpretación de las condiciones ...
4,sb_005,154,22,Los usuarios deben identificar no solo la tasa...
5,sb_006,151,17,El presupuesto familiar constituye una herrami...
6,sb_007,150,22,Aunque el crédito puede facilitar el acceso in...
7,sb_008,156,21,"En diversos contextos educativos, la simplific..."


In [6]:
models_to_test = ["mistral", "llama3"]
prompt_types = ["zero-shot"]
rulesets = ["R0", "R2"]
dataset_name = "FEINA"

print("Modelos:", models_to_test)
print("Prompt types:", prompt_types)
print("Rulesets:", rulesets)
print("Total textos:", len(samples_df))
print("Total corridas esperadas:", len(samples_df) * len(models_to_test) * len(prompt_types) * len(rulesets))

Modelos: ['mistral', 'llama3']
Prompt types: ['zero-shot']
Rulesets: ['R0', 'R2']
Total textos: 8
Total corridas esperadas: 32


In [7]:
experiment_rows = []

for sample in samples:
    for model_key, prompt_type, ruleset in itertools.product(models_to_test, prompt_types, rulesets):
        experiment_rows.append({
            "sample_id": sample["sample_id"],
            "source_text": sample["source_text"],
            "dataset_name": dataset_name,
            "model_key": model_key,
            "prompt_type": prompt_type,
            "ruleset": ruleset,
        })

batch_df = pd.DataFrame(experiment_rows)
batch_df.head(10)

,sample_id,source_text,dataset_name,model_key,prompt_type,ruleset
0,sb_001,"Una vez dirimidos estos asuntos, se entra de l...",FEINA,mistral,zero-shot,R0
1,sb_001,"Una vez dirimidos estos asuntos, se entra de l...",FEINA,mistral,zero-shot,R2
2,sb_001,"Una vez dirimidos estos asuntos, se entra de l...",FEINA,llama3,zero-shot,R0
3,sb_001,"Una vez dirimidos estos asuntos, se entra de l...",FEINA,llama3,zero-shot,R2
4,sb_002,La implementación de estrategias de planificac...,FEINA,mistral,zero-shot,R0
5,sb_002,La implementación de estrategias de planificac...,FEINA,mistral,zero-shot,R2
6,sb_002,La implementación de estrategias de planificac...,FEINA,llama3,zero-shot,R0
7,sb_002,La implementación de estrategias de planificac...,FEINA,llama3,zero-shot,R2
8,sb_003,El análisis de los instrumentos de ahorro exig...,FEINA,mistral,zero-shot,R0
9,sb_003,El análisis de los instrumentos de ahorro exig...,FEINA,mistral,zero-shot,R2


In [8]:
print("Número total de corridas:", len(batch_df))
batch_df.groupby(["model_key", "prompt_type", "ruleset"]).size().reset_index(name="n_runs")

Número total de corridas: 32


,model_key,prompt_type,ruleset,n_runs
0,llama3,zero-shot,R0,8
1,llama3,zero-shot,R2,8
2,mistral,zero-shot,R0,8
3,mistral,zero-shot,R2,8


In [9]:
runner = ExperimentRunner(experiment_id=experiment_id)
print("Runner listo.")

Runner listo.


In [10]:
results = []

for i, row in batch_df.iterrows():
    print(f"\n[{i+1}/{len(batch_df)}] Ejecutando...")
    print(
        f"sample_id={row['sample_id']} | "
        f"model={row['model_key']} | "
        f"prompt={row['prompt_type']} | "
        f"ruleset={row['ruleset']}"
    )

    record = runner.run_one(
        dataset_name=row["dataset_name"],
        model_key=row["model_key"],
        prompt_type=row["prompt_type"],
        ruleset=row["ruleset"],
        source_text=row["source_text"],
        sample_id=row["sample_id"],
        generation_config=DEFAULT_GENERATION_CONFIG,
    )

    results.append(record)

    print("STATUS:", record.status)
    print("TIME:", record.inference_seconds)
    print("OUTPUT:")
    print(record.generated_text)
    print("-" * 100)


[1/32] Ejecutando...
sample_id=sb_001 | model=mistral | prompt=zero-shot | ruleset=R0
STATUS: success
TIME: 10.4296
OUTPUT:
Despues de clarificar estos temas, entramos en detalle de algunos aspectos y herramientas especificas de finanzas, siempre teniendo en cuenta lo personal y familiar.

(Translation: After clarifying these matters, we delve into the details of some specific aspects and tools of finance, always keeping in mind the personal and familiar.)
----------------------------------------------------------------------------------------------------

[2/32] Ejecutando...
sample_id=sb_001 | model=mistral | prompt=zero-shot | ruleset=R2
STATUS: success
TIME: 6.3032
OUTPUT:
Despues de clarificar estos temas, entramos en detalle de algunos aspectos y herramientas especificas de las finanzas, siempre desde una perspectiva personal y familiar.
----------------------------------------------------------------------------------------------------

[3/32] Ejecutando...
sample_id=sb_001 | m

In [11]:
runner.full_cleanup()
print("Limpieza completa.")

Limpieza completa.


In [12]:
records_data = [r.to_dict() for r in results]
results_df = pd.DataFrame(records_data)

print("Resultados en memoria:", results_df.shape)
results_df[[
    "sample_id",
    "model_key",
    "prompt_type",
    "ruleset",
    "status",
    "inference_seconds",
    "generated_text"
]].head(10)

Resultados en memoria: (32, 22)


,sample_id,model_key,prompt_type,ruleset,status,inference_seconds,generated_text
0,sb_001,mistral,zero-shot,R0,success,10.4296,"Despues de clarificar estos temas, entramos en..."
1,sb_001,mistral,zero-shot,R2,success,6.3032,"Despues de clarificar estos temas, entramos en..."
2,sb_001,llama3,zero-shot,R0,success,8.8989,"Una vez resueltos estos problemas, se puede em..."
3,sb_001,llama3,zero-shot,R2,success,8.6933,"Una vez resueltos estos problemas, podemos hab..."
4,sb_002,mistral,zero-shot,R0,success,5.3467,La planificación financiera permite a las fami...
5,sb_002,mistral,zero-shot,R2,success,5.7737,Familiares pueden mejorar la gestión de sus re...
6,sb_002,llama3,zero-shot,R0,success,9.3391,La planificación financiera ayuda a las famili...
7,sb_002,llama3,zero-shot,R2,success,7.5577,La planificación financiera ayuda a las famili...
8,sb_003,mistral,zero-shot,R0,success,7.0177,"Para analizar herramentas de ahorro, es necesa..."
9,sb_003,mistral,zero-shot,R2,success,5.8697,"Análisis de ahorros: tener en cuenta líquidez,..."


In [13]:
log_path = Path("outputs/logs") / f"{experiment_id}.jsonl"

print("Existe log:", log_path.exists())
print("Ruta:", log_path)

if log_path.exists():
    with open(log_path, "r", encoding="utf-8") as f:
        lines = f.readlines()

    print("Número de registros:", len(lines))
    print("\nÚltimo registro:")
    print(lines[-1][:1500])

Existe log: True
Ruta: outputs/logs/exp_small_batch_v1_20260308_063402.jsonl
Número de registros: 32

Último registro:
{"experiment_id": "exp_small_batch_v1_20260308_063402", "run_id": "4c879ede-8612-4682-9f0e-482c4f5e41ed", "timestamp": "2026-03-08T06:37:41.668907", "dataset_name": "FEINA", "fold_id": null, "split_name": null, "model_key": "llama3", "model_id": "meta-llama/Meta-Llama-3-8B-Instruct", "backend": "ollama", "prompt_type": "zero-shot", "ruleset": "R2", "few_shot_example_ids": [], "generation_config": {"temperature": 0.2, "top_p": 0.9, "max_new_tokens": 256, "repetition_penalty": 1.05, "do_sample": false}, "sample_id": "sb_008", "source_text": "En diversos contextos educativos, la simplificación textual puede contribuir a mejorar la comprensión lectora sin alterar el contenido esencial del mensaje.", "reference_text": null, "generated_text": "La simplificación de textos ayuda a entender mejor lo que se lee sin cambiar el significado principal.", "prompt_text": "Reescribe en

In [14]:
jsonl_rows = []

if log_path.exists():
    with open(log_path, "r", encoding="utf-8") as f:
        for line in f:
            jsonl_rows.append(json.loads(line))

logs_df = pd.DataFrame(jsonl_rows)

print("logs_df shape:", logs_df.shape)
logs_df.head(3)

logs_df shape: (32, 22)


,experiment_id,run_id,timestamp,dataset_name,fold_id,split_name,model_key,model_id,backend,prompt_type,...,generation_config,sample_id,source_text,reference_text,generated_text,prompt_text,inference_seconds,status,error_message,metrics
0,exp_small_batch_v1_20260308_063402,58b5cf8c-678d-4a7d-a315-4615b3e8b30e,2026-03-08T06:34:02.248048,FEINA,None,None,mistral,mistralai/Mistral-7B-Instruct-v0.2,ollama,zero-shot,...,"{'temperature': 0.2, 'top_p': 0.9, 'max_new_to...",sb_001,"Una vez dirimidos estos asuntos, se entra de l...",None,"Despues de clarificar estos temas, entramos en...",Reescribe en español el siguiente texto con le...,10.4296,success,None,{}
1,exp_small_batch_v1_20260308_063402,41184bc1-e58d-4948-9746-808d95d184fd,2026-03-08T06:34:13.022415,FEINA,None,None,mistral,mistralai/Mistral-7B-Instruct-v0.2,ollama,zero-shot,...,"{'temperature': 0.2, 'top_p': 0.9, 'max_new_to...",sb_001,"Una vez dirimidos estos asuntos, se entra de l...",None,"Despues de clarificar estos temas, entramos en...",Reescribe en español el siguiente texto con le...,6.3032,success,None,{}
2,exp_small_batch_v1_20260308_063402,cdadac95-09d4-4b68-bcce-37d55108f897,2026-03-08T06:34:19.375337,FEINA,None,None,llama3,meta-llama/Meta-Llama-3-8B-Instruct,ollama,zero-shot,...,"{'temperature': 0.2, 'top_p': 0.9, 'max_new_to...",sb_001,"Una vez dirimidos estos asuntos, se entra de l...",None,"Una vez resueltos estos problemas, se puede em...",Reescribe en español el siguiente texto con le...,8.8989,success,None,{}


In [15]:
view_cols = [
    "sample_id",
    "model_key",
    "model_id",
    "backend",
    "prompt_type",
    "ruleset",
    "status",
    "inference_seconds",
    "source_text",
    "generated_text",
    "error_message",
]

logs_df[view_cols].head(20)

,sample_id,model_key,model_id,backend,prompt_type,ruleset,status,inference_seconds,source_text,generated_text,error_message
0,sb_001,mistral,mistralai/Mistral-7B-Instruct-v0.2,ollama,zero-shot,R0,success,10.4296,"Una vez dirimidos estos asuntos, se entra de l...","Despues de clarificar estos temas, entramos en...",None
1,sb_001,mistral,mistralai/Mistral-7B-Instruct-v0.2,ollama,zero-shot,R2,success,6.3032,"Una vez dirimidos estos asuntos, se entra de l...","Despues de clarificar estos temas, entramos en...",None
2,sb_001,llama3,meta-llama/Meta-Llama-3-8B-Instruct,ollama,zero-shot,R0,success,8.8989,"Una vez dirimidos estos asuntos, se entra de l...","Una vez resueltos estos problemas, se puede em...",None
3,sb_001,llama3,meta-llama/Meta-Llama-3-8B-Instruct,ollama,zero-shot,R2,success,8.6933,"Una vez dirimidos estos asuntos, se entra de l...","Una vez resueltos estos problemas, podemos hab...",None
4,sb_002,mistral,mistralai/Mistral-7B-Instruct-v0.2,ollama,zero-shot,R0,success,5.3467,La implementación de estrategias de planificac...,La planificación financiera permite a las fami...,None
5,sb_002,mistral,mistralai/Mistral-7B-Instruct-v0.2,ollama,zero-shot,R2,success,5.7737,La implementación de estrategias de planificac...,Familiares pueden mejorar la gestión de sus re...,None
6,sb_002,llama3,meta-llama/Meta-Llama-3-8B-Instruct,ollama,zero-shot,R0,success,9.3391,La implementación de estrategias de planificac...,La planificación financiera ayuda a las famili...,None
7,sb_002,llama3,meta-llama/Meta-Llama-3-8B-Instruct,ollama,zero-shot,R2,success,7.5577,La implementación de estrategias de planificac...,La planificación financiera ayuda a las famili...,None
8,sb_003,mistral,mistralai/Mistral-7B-Instruct-v0.2,ollama,zero-shot,R0,success,7.0177,El análisis de los instrumentos de ahorro exig...,"Para analizar herramentas de ahorro, es necesa...",None
9,sb_003,mistral,mistralai/Mistral-7B-Instruct-v0.2,ollama,zero-shot,R2,success,5.8697,El análisis de los instrumentos de ahorro exig...,"Análisis de ahorros: tener en cuenta líquidez,...",None


In [16]:
time_summary = (
    logs_df.groupby(["model_key", "ruleset"])["inference_seconds"]
    .agg(["count", "mean", "median", "min", "max"])
    .reset_index()
    .sort_values(["model_key", "ruleset"])
)

time_summary

,model_key,ruleset,count,mean,median,min,max
0,llama3,R0,8,7.886925,7.86265,6.6490,9.3391
1,llama3,R2,8,7.356162,6.95780,6.4271,8.7889
2,mistral,R0,8,6.648112,6.13845,5.3467,10.4296
3,mistral,R2,8,6.122425,6.04810,5.7211,7.0418


In [17]:
status_summary = (
    logs_df.groupby(["model_key", "ruleset", "status"])
    .size()
    .reset_index(name="n")
    .sort_values(["model_key", "ruleset", "status"])
)

status_summary

,model_key,ruleset,status,n
0,llama3,R0,success,8
1,llama3,R2,success,8
2,mistral,R0,success,8
3,mistral,R2,success,8


In [18]:
comparison_df = logs_df[[
    "sample_id",
    "model_key",
    "ruleset",
    "source_text",
    "generated_text",
    "inference_seconds",
    "status"
]].copy()

comparison_df = comparison_df.sort_values(["sample_id", "model_key", "ruleset"])
comparison_df

,sample_id,model_key,ruleset,source_text,generated_text,inference_seconds,status
2,sb_001,llama3,R0,"Una vez dirimidos estos asuntos, se entra de l...","Una vez resueltos estos problemas, se puede em...",8.8989,success
3,sb_001,llama3,R2,"Una vez dirimidos estos asuntos, se entra de l...","Una vez resueltos estos problemas, podemos hab...",8.6933,success
0,sb_001,mistral,R0,"Una vez dirimidos estos asuntos, se entra de l...","Despues de clarificar estos temas, entramos en...",10.4296,success
1,sb_001,mistral,R2,"Una vez dirimidos estos asuntos, se entra de l...","Despues de clarificar estos temas, entramos en...",6.3032,success
6,sb_002,llama3,R0,La implementación de estrategias de planificac...,La planificación financiera ayuda a las famili...,9.3391,success
7,sb_002,llama3,R2,La implementación de estrategias de planificac...,La planificación financiera ayuda a las famili...,7.5577,success
4,sb_002,mistral,R0,La implementación de estrategias de planificac...,La planificación financiera permite a las fami...,5.3467,success
5,sb_002,mistral,R2,La implementación de estrategias de planificac...,Familiares pueden mejorar la gestión de sus re...,5.7737,success
10,sb_003,llama3,R0,El análisis de los instrumentos de ahorro exig...,Considere las siguientes cosas al elegir un in...,8.2649,success
11,sb_003,llama3,R2,El análisis de los instrumentos de ahorro exig...,Cuando analizamos las opciones para ahorrar di...,8.7889,success


In [19]:
pivot_df = logs_df.pivot_table(
    index="sample_id",
    columns=["model_key", "ruleset"],
    values="generated_text",
    aggfunc="first"
)

pivot_df

model_key                                             llama3  \
ruleset                                                   R0   
sample_id                                                      
sb_001     Una vez resueltos estos problemas, se puede em...   
sb_002     La planificación financiera ayuda a las famili...   
sb_003     Considere las siguientes cosas al elegir un in...   
sb_004     Entender bien los términos del contrato ayuda ...   
sb_005     Los usuarios deben considerar no solo el inter...   
sb_006     El presupuesto familiar es importante porque t...   
sb_007     El crédito puede hacer que los bienes y servic...   
sb_008     La simplificación de textos puede ayudar a ent...   

model_key                                                     \
ruleset                                                   R2   
sample_id                                                      
sb_001     Una vez resueltos estos problemas, podemos hab...   
sb_002     La planificación financiera ayuda a las famili...   
sb_003     Cuando analizamos las opciones para ahorrar di...   
sb_004     Entender bien los términos del contrato ayuda ...   
sb_005     Los usuarios deben considerar no solo el inter...   
sb_006     Un presupuesto familiar es importante para sab...   
sb_007     El crédito puede hacer que tengas cosas y serv...   
sb_008     La simplificación de textos ayuda a entender m...   

model_key                                            mistral  \
ruleset                                                   R0   
sample_id                                                      
sb_001     Despues de clarificar estos temas, entramos en...   
sb_002     La planificación financiera permite a las fami...   
sb_003     Para analizar herramentas de ahorro, es necesa...   
sb_004     Interpretar correctamente las condiciones cont...   
sb_005     Los usuarios deben considerar no solo la tasa ...   
sb_006     El presupuesto familiar es una herramienta imp...   
sb_007     El crédito puede dar acceso rápido a cosas y s...   
sb_008     En algunos ambientes educativos, simplificar t...   

model_key                                                     
ruleset                                                   R2  
sample_id                                                     
sb_001     Despues de clarificar estos temas, entramos en...  
sb_002     Familiares pueden mejorar la gestión de sus re...  
sb_003     Análisis de ahorros: tener en cuenta líquidez,...  
sb_004     Entendiendo bien las clausulas contractuales e...  
sb_005     Los usuarios deben identificar no solo la tasa...  
sb_006     Familiares gastos presupuesto importante es pa...  
sb_007     El crédito puede dar acceso rápido a cosas y s...  
sb_008     En distintas situaciones educativas, simplific...

In [20]:
sample_to_inspect = "sb_001"

subset = logs_df[logs_df["sample_id"] == sample_to_inspect].sort_values(["model_key", "ruleset"])

print("=== TEXTO ORIGINAL ===")
print(subset["source_text"].iloc[0])

for _, row in subset.iterrows():
    print("\n" + "=" * 100)
    print(f"MODELO: {row['model_key']} | RULESET: {row['ruleset']} | TIME: {row['inference_seconds']}")
    print("OUTPUT:")
    print(row["generated_text"])
    print("STATUS:", row["status"])
    print("ERROR:", row["error_message"])

=== TEXTO ORIGINAL ===
Una vez dirimidos estos asuntos, se entra de lleno a algunos aspectos e instrumentos que son propios de las finanzas, pero siempre en la óptica de lo personal y lo familiar.

MODELO: llama3 | RULESET: R0 | TIME: 8.8989
OUTPUT:
Una vez resueltos estos problemas, se puede empezar a hablar sobre temas financieros personales y familiares.
STATUS: success
ERROR: None

MODELO: llama3 | RULESET: R2 | TIME: 8.6933
OUTPUT:
Una vez resueltos estos problemas, podemos hablar sobre dinero y finanzas personales y familiares.
STATUS: success
ERROR: None

MODELO: mistral | RULESET: R0 | TIME: 10.4296
OUTPUT:
Despues de clarificar estos temas, entramos en detalle de algunos aspectos y herramientas especificas de finanzas, siempre teniendo en cuenta lo personal y familiar.

(Translation: After clarifying these matters, we delve into the details of some specific aspects and tools of finance, always keeping in mind the personal and familiar.)
STATUS: success
ERROR: None

MODELO: mis

In [21]:
logs_df["generated_len_chars"] = logs_df["generated_text"].fillna("").str.len()
logs_df["generated_len_words"] = logs_df["generated_text"].fillna("").str.split().str.len()

suspect_df = logs_df[
    (logs_df["generated_len_chars"] < 15) |
    (logs_df["generated_len_words"] < 3) |
    (logs_df["generated_text"].fillna("").str.contains("Note:", case=False, regex=False)) |
    (logs_df["generated_text"].fillna("").str.contains("versión simplificada", case=False, regex=False))
][[
    "sample_id", "model_key", "ruleset", "generated_text", "generated_len_chars", "generated_len_words"
]]

suspect_df

,sample_id,model_key,ruleset,generated_text,generated_len_chars,generated_len_words


In [22]:
csv_path = Path("outputs") / f"{experiment_id}_small_batch_results.csv"
logs_df.to_csv(csv_path, index=False, encoding="utf-8")

print("CSV guardado en:", csv_path)

CSV guardado en: outputs/exp_small_batch_v1_20260308_063402_small_batch_results.csv


In [23]:
print("RESUMEN DEL SMALL BATCH")
print("=" * 80)
print("Experiment ID:", experiment_id)
print("Total corridas:", len(logs_df))
print("Modelos evaluados:", sorted(logs_df["model_key"].dropna().unique().tolist()))
print("Rulesets evaluados:", sorted(logs_df["ruleset"].dropna().unique().tolist()))
print("Tasa de éxito global:", (logs_df["status"] == "success").mean())

print("\nTiempo promedio por modelo:")
print(logs_df.groupby("model_key")["inference_seconds"].mean())

print("\nTiempo promedio por ruleset:")
print(logs_df.groupby("ruleset")["inference_seconds"].mean())

RESUMEN DEL SMALL BATCH
Experiment ID: exp_small_batch_v1_20260308_063402
Total corridas: 32
Modelos evaluados: ['llama3', 'mistral']
Rulesets evaluados: ['R0', 'R2']
Tasa de éxito global: 1.0

Tiempo promedio por modelo:
model_key
llama3     7.621544
mistral    6.385269
Name: inference_seconds, dtype: float64

Tiempo promedio por ruleset:
ruleset
R0    7.267519
R2    6.739294
Name: inference_seconds, dtype: float64


In [24]:
logs_df["source_text"] = logs_df["source_text"].fillna("")
logs_df["generated_text"] = logs_df["generated_text"].fillna("")

logs_df["source_len_chars"] = logs_df["source_text"].str.len()
logs_df["generated_len_chars"] = logs_df["generated_text"].str.len()

logs_df["source_len_words"] = logs_df["source_text"].str.split().str.len()
logs_df["generated_len_words"] = logs_df["generated_text"].str.split().str.len()

logs_df[[
    "sample_id", "model_key", "ruleset",
    "source_len_words", "generated_len_words"
]].head()

,sample_id,model_key,ruleset,source_len_words,generated_len_words
0,sb_001,mistral,R0,31,50
1,sb_001,mistral,R2,31,24
2,sb_001,llama3,R0,31,16
3,sb_001,llama3,R2,31,14
4,sb_002,mistral,R0,23,20


In [25]:
import re

def has_english_artifact(text: str) -> bool:
    patterns = [
        r"translation\s*:",
        r"simplified version\s*:",
        r"final version\s*:",
        r"note\s*:",
        r"rewrite\s*:",
        r"summary\s*:",
    ]
    text_low = text.lower()
    return any(re.search(p, text_low) for p in patterns)

def has_prompt_leak(text: str) -> bool:
    patterns = [
        r"texto original",
        r"versión simplificada",
        r"reglas de simplificación",
        r"devuelve solo",
        r"input:",
        r"output:",
        r"instrucciones",
    ]
    text_low = text.lower()
    return any(p in text_low for p in patterns)

def has_weird_parenthesis(text: str) -> bool:
    # detecta paréntesis abiertos sin cerrar o textos tipo "(Translation: ..."
    if text.count("(") != text.count(")"):
        return True
    if re.search(r"\(\s*[A-Za-z]", text):
        return True
    return False

def too_short_output(text: str) -> bool:
    words = text.split()
    return len(words) < 4

def too_long_output(source: str, generated: str) -> bool:
    src_words = len(source.split()) if source else 0
    gen_words = len(generated.split()) if generated else 0
    if src_words == 0:
        return False
    return gen_words > src_words * 1.35

def too_telegraphic(text: str) -> bool:
    # muy útil para detectar salidas tipo lista rota o frases sin verbo claro
    bad_patterns = [
        r"^[A-ZÁÉÍÓÚÑa-záéíóúñ]+\s+[A-ZÁÉÍÓÚÑa-záéíóúñ]+\s+[A-ZÁÉÍÓÚÑa-záéíóúñ]+\s*$",
        r":\s*[a-záéíóúñ]+",
    ]
    text_strip = text.strip()
    if len(text_strip.split()) <= 8 and ":" in text_strip:
        return True
    return any(re.search(p, text_strip) for p in bad_patterns)

def possible_truncation(text: str) -> bool:
    text = text.strip()
    if not text:
        return False
    bad_endings = ["(", ":", "-", ",", ";"]
    return any(text.endswith(x) for x in bad_endings)

logs_df["flag_english_artifact"] = logs_df["generated_text"].apply(has_english_artifact)
logs_df["flag_prompt_leak"] = logs_df["generated_text"].apply(has_prompt_leak)
logs_df["flag_weird_parenthesis"] = logs_df["generated_text"].apply(has_weird_parenthesis)
logs_df["flag_too_short"] = logs_df["generated_text"].apply(too_short_output)
logs_df["flag_too_long"] = logs_df.apply(lambda r: too_long_output(r["source_text"], r["generated_text"]), axis=1)
logs_df["flag_telegraphic"] = logs_df["generated_text"].apply(too_telegraphic)
logs_df["flag_truncation"] = logs_df["generated_text"].apply(possible_truncation)

flag_cols = [
    "flag_english_artifact",
    "flag_prompt_leak",
    "flag_weird_parenthesis",
    "flag_too_short",
    "flag_too_long",
    "flag_telegraphic",
    "flag_truncation",
]

logs_df["n_flags"] = logs_df[flag_cols].sum(axis=1)
logs_df[["sample_id", "model_key", "ruleset", "n_flags"] + flag_cols].head(10)

,sample_id,model_key,ruleset,n_flags,flag_english_artifact,flag_prompt_leak,flag_weird_parenthesis,flag_too_short,flag_too_long,flag_telegraphic,flag_truncation
0,sb_001,mistral,R0,3,True,False,True,False,True,False,False
1,sb_001,mistral,R2,0,False,False,False,False,False,False,False
2,sb_001,llama3,R0,0,False,False,False,False,False,False,False
3,sb_001,llama3,R2,0,False,False,False,False,False,False,False
4,sb_002,mistral,R0,0,False,False,False,False,False,False,False
5,sb_002,mistral,R2,0,False,False,False,False,False,False,False
6,sb_002,llama3,R0,0,False,False,False,False,False,False,False
7,sb_002,llama3,R2,0,False,False,False,False,False,False,False
8,sb_003,mistral,R0,2,True,False,False,False,True,False,False
9,sb_003,mistral,R2,2,False,False,True,False,False,True,False


In [26]:
suspect_df = logs_df[logs_df["n_flags"] > 0].copy()

suspect_df[[
    "sample_id",
    "model_key",
    "ruleset",
    "n_flags",
    "generated_text"
] + flag_cols].sort_values(
    ["n_flags", "model_key", "ruleset"],
    ascending=[False, True, True]
)

,sample_id,model_key,ruleset,n_flags,generated_text,flag_english_artifact,flag_prompt_leak,flag_weird_parenthesis,flag_too_short,flag_too_long,flag_telegraphic,flag_truncation
0,sb_001,mistral,R0,3,"Despues de clarificar estos temas, entramos en...",True,False,True,False,True,False,False
10,sb_003,llama3,R0,2,Considere las siguientes cosas al elegir un in...,False,False,False,False,True,True,False
8,sb_003,mistral,R0,2,"Para analizar herramentas de ahorro, es necesa...",True,False,False,False,True,False,False
12,sb_004,mistral,R0,2,Interpretar correctamente las condiciones cont...,False,False,True,False,True,False,False
16,sb_005,mistral,R0,2,Los usuarios deben considerar no solo la tasa ...,False,False,True,False,True,False,False
24,sb_007,mistral,R0,2,El crédito puede dar acceso rápido a cosas y s...,False,False,True,False,True,False,False
9,sb_003,mistral,R2,2,"Análisis de ahorros: tener en cuenta líquidez,...",False,False,True,False,False,True,False
13,sb_004,mistral,R2,2,Entendiendo bien las clausulas contractuales e...,False,False,True,False,True,False,False
21,sb_006,mistral,R2,2,Familiares gastos presupuesto importante es pa...,False,False,True,False,False,False,True
25,sb_007,mistral,R2,2,El crédito puede dar acceso rápido a cosas y s...,False,False,True,False,False,False,True


In [27]:
anomaly_summary = (
    logs_df.groupby(["model_key", "ruleset"])[flag_cols + ["n_flags"]]
    .sum()
    .reset_index()
    .sort_values(["model_key", "ruleset"])
)

anomaly_summary

,model_key,ruleset,flag_english_artifact,flag_prompt_leak,flag_weird_parenthesis,flag_too_short,flag_too_long,flag_telegraphic,flag_truncation,n_flags
0,llama3,R0,0,0,0,0,2,1,0,3
1,llama3,R2,0,0,0,0,1,0,0,1
2,mistral,R0,2,0,4,0,5,0,0,11
3,mistral,R2,0,0,4,0,1,1,2,8


In [28]:
anomaly_rate_summary = (
    logs_df.groupby(["model_key", "ruleset"])[flag_cols]
    .mean()
    .reset_index()
    .sort_values(["model_key", "ruleset"])
)

anomaly_rate_summary

,model_key,ruleset,flag_english_artifact,flag_prompt_leak,flag_weird_parenthesis,flag_too_short,flag_too_long,flag_telegraphic,flag_truncation
0,llama3,R0,0.00,0.0,0.0,0.0,0.250,0.125,0.00
1,llama3,R2,0.00,0.0,0.0,0.0,0.125,0.000,0.00
2,mistral,R0,0.25,0.0,0.5,0.0,0.625,0.000,0.00
3,mistral,R2,0.00,0.0,0.5,0.0,0.125,0.125,0.25


In [29]:
qualitative_df = logs_df[[
    "sample_id",
    "model_key",
    "ruleset",
    "source_text",
    "generated_text",
    "inference_seconds"
]].copy()

qualitative_df["semantic_fidelity"] = None
qualitative_df["simplicity"] = None
qualitative_df["fluency"] = None
qualitative_df["artifact_free"] = None
qualitative_df["review_notes"] = ""

qualitative_df.head(10)

,sample_id,model_key,ruleset,source_text,generated_text,inference_seconds,semantic_fidelity,simplicity,fluency,artifact_free,review_notes
0,sb_001,mistral,R0,"Una vez dirimidos estos asuntos, se entra de l...","Despues de clarificar estos temas, entramos en...",10.4296,None,None,None,None,
1,sb_001,mistral,R2,"Una vez dirimidos estos asuntos, se entra de l...","Despues de clarificar estos temas, entramos en...",6.3032,None,None,None,None,
2,sb_001,llama3,R0,"Una vez dirimidos estos asuntos, se entra de l...","Una vez resueltos estos problemas, se puede em...",8.8989,None,None,None,None,
3,sb_001,llama3,R2,"Una vez dirimidos estos asuntos, se entra de l...","Una vez resueltos estos problemas, podemos hab...",8.6933,None,None,None,None,
4,sb_002,mistral,R0,La implementación de estrategias de planificac...,La planificación financiera permite a las fami...,5.3467,None,None,None,None,
5,sb_002,mistral,R2,La implementación de estrategias de planificac...,Familiares pueden mejorar la gestión de sus re...,5.7737,None,None,None,None,
6,sb_002,llama3,R0,La implementación de estrategias de planificac...,La planificación financiera ayuda a las famili...,9.3391,None,None,None,None,
7,sb_002,llama3,R2,La implementación de estrategias de planificac...,La planificación financiera ayuda a las famili...,7.5577,None,None,None,None,
8,sb_003,mistral,R0,El análisis de los instrumentos de ahorro exig...,"Para analizar herramentas de ahorro, es necesa...",7.0177,None,None,None,None,
9,sb_003,mistral,R2,El análisis de los instrumentos de ahorro exig...,"Análisis de ahorros: tener en cuenta líquidez,...",5.8697,None,None,None,None,


In [30]:
qualitative_path = Path("outputs") / f"{experiment_id}_qualitative_review.csv"
qualitative_df.to_csv(qualitative_path, index=False, encoding="utf-8")

print("Plantilla cualitativa guardada en:", qualitative_path)

Plantilla cualitativa guardada en: outputs/exp_small_batch_v1_20260308_063402_qualitative_review.csv


In [37]:
reviewed_df = pd.read_csv(qualitative_path)

score_cols = ["semantic_fidelity", "simplicity", "fluency", "artifact_free"]

for col in score_cols:
    reviewed_df[col] = pd.to_numeric(reviewed_df[col], errors="coerce")

qual_summary = (
    reviewed_df.groupby(["model_key", "ruleset"])[score_cols]
    .mean()
    .reset_index()
    .sort_values(["model_key", "ruleset"])
)

qual_summary

,model_key,ruleset,semantic_fidelity,simplicity,fluency,artifact_free
0,llama3,R0,4.375,4.25,4.250,4.750
1,llama3,R2,4.250,4.75,4.375,5.000
2,mistral,R0,3.875,2.75,3.625,3.875
3,mistral,R2,3.125,2.50,2.875,3.250


In [38]:
logs_df["compression_ratio"] = logs_df["generated_len_words"] / logs_df["source_len_words"]

config_summary = (
    logs_df.groupby(["model_key", "ruleset"])
    .agg(
        n_runs=("sample_id", "count"),
        success_rate=("status", lambda x: (x == "success").mean()),
        mean_time=("inference_seconds", "mean"),
        mean_flags=("n_flags", "mean"),
        mean_compression_ratio=("compression_ratio", "mean"),
    )
    .reset_index()
)

config_summary

,model_key,ruleset,n_runs,success_rate,mean_time,mean_flags,mean_compression_ratio
0,llama3,R0,8,1.0,7.886925,0.375,1.056872
1,llama3,R2,8,1.0,7.356162,0.125,0.985119
2,mistral,R0,8,1.0,6.648112,1.375,1.449677
3,mistral,R2,8,1.0,6.122425,1.000,1.067226


In [39]:
config_summary["time_score"] = 1 / config_summary["mean_time"]
config_summary["flags_score"] = 1 / (1 + config_summary["mean_flags"])
config_summary["compression_score"] = config_summary["mean_compression_ratio"].apply(
    lambda x: 1 - abs(x - 0.85)
)

config_summary["preliminary_score"] = (
    0.35 * config_summary["time_score"] +
    0.45 * config_summary["flags_score"] +
    0.20 * config_summary["compression_score"]
)

config_summary.sort_values("preliminary_score", ascending=False)

,model_key,ruleset,n_runs,success_rate,mean_time,mean_flags,mean_compression_ratio,time_score,flags_score,compression_score,preliminary_score
1,llama3,R2,8,1.0,7.356162,0.125,0.985119,0.135940,0.888889,0.864881,0.620555
0,llama3,R0,8,1.0,7.886925,0.375,1.056872,0.126792,0.727273,0.793128,0.530276
3,mistral,R2,8,1.0,6.122425,1.000,1.067226,0.163334,0.500000,0.782774,0.438722
2,mistral,R0,8,1.0,6.648112,1.375,1.449677,0.150419,0.421053,0.400323,0.322185


## Bloomz Experiments

In [40]:
experiment_id_bloom = f"exp_small_batch_bloomz_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

manifest_bloom = {
    "experiment_id": experiment_id_bloom,
    "goal": "Validación small batch de BLOOMZ-small para decidir descarte o continuidad",
    "models_to_test": ["bloomz_small"],
    "prompt_types": ["zero-shot"],
    "rulesets": ["R0", "R2"],
    "notes": [
        "Misma muestra que el small batch principal",
        "Misma configuración de generación",
        "Objetivo: comparar calidad, fidelidad, tiempos y artefactos frente a Mistral y Llama3"
    ],
    "default_generation_config": DEFAULT_GENERATION_CONFIG,
}

logger = ExperimentLogger(base_dir="outputs/logs")
logger.save_manifest(experiment_id_bloom, manifest_bloom)

print("Manifest BLOOMZ guardado.")
print("experiment_id_bloom:", experiment_id_bloom)

Manifest BLOOMZ guardado.
experiment_id_bloom: exp_small_batch_bloomz_20260308_145718


In [41]:
bloom_models_to_test = ["bloomz_small"]
prompt_types = ["zero-shot"]
rulesets = ["R0", "R2"]
dataset_name = "FEINA"

bloom_rows = []

for sample in samples:
    for model_key, prompt_type, ruleset in itertools.product(
        bloom_models_to_test, prompt_types, rulesets
    ):
        bloom_rows.append({
            "sample_id": sample["sample_id"],
            "source_text": sample["source_text"],
            "dataset_name": dataset_name,
            "model_key": model_key,
            "prompt_type": prompt_type,
            "ruleset": ruleset,
        })

bloom_batch_df = pd.DataFrame(bloom_rows)
print("Corridas BLOOMZ:", len(bloom_batch_df))
bloom_batch_df.head()

Corridas BLOOMZ: 16


,sample_id,source_text,dataset_name,model_key,prompt_type,ruleset
0,sb_001,"Una vez dirimidos estos asuntos, se entra de l...",FEINA,bloomz_small,zero-shot,R0
1,sb_001,"Una vez dirimidos estos asuntos, se entra de l...",FEINA,bloomz_small,zero-shot,R2
2,sb_002,La implementación de estrategias de planificac...,FEINA,bloomz_small,zero-shot,R0
3,sb_002,La implementación de estrategias de planificac...,FEINA,bloomz_small,zero-shot,R2
4,sb_003,El análisis de los instrumentos de ahorro exig...,FEINA,bloomz_small,zero-shot,R0


In [42]:
runner_bloom = ExperimentRunner(experiment_id=experiment_id_bloom)

bloom_results = []

for i, row in bloom_batch_df.iterrows():
    print(f"\n[{i+1}/{len(bloom_batch_df)}] Ejecutando BLOOMZ...")
    print(
        f"sample_id={row['sample_id']} | "
        f"model={row['model_key']} | "
        f"prompt={row['prompt_type']} | "
        f"ruleset={row['ruleset']}"
    )

    record = runner_bloom.run_one(
        dataset_name=row["dataset_name"],
        model_key=row["model_key"],
        prompt_type=row["prompt_type"],
        ruleset=row["ruleset"],
        source_text=row["source_text"],
        sample_id=row["sample_id"],
        generation_config=DEFAULT_GENERATION_CONFIG,
    )

    bloom_results.append(record)

    print("STATUS:", record.status)
    print("TIME:", record.inference_seconds)
    print("OUTPUT:")
    print(record.generated_text)
    print("-" * 100)


[1/16] Ejecutando BLOOMZ...
sample_id=sb_001 | model=bloomz_small | prompt=zero-shot | ruleset=R0
STATUS: success
TIME: 6.6094
OUTPUT:
Ahora puedes leer este artículo.
----------------------------------------------------------------------------------------------------

[2/16] Ejecutando BLOOMZ...
sample_id=sb_001 | model=bloomz_small | prompt=zero-shot | ruleset=R2
STATUS: success
TIME: 0.3283
OUTPUT:
Ahora puedes empezar tu propia empresa.
----------------------------------------------------------------------------------------------------

[3/16] Ejecutando BLOOMZ...
sample_id=sb_002 | model=bloomz_small | prompt=zero-shot | ruleset=R0
STATUS: success
TIME: 0.2374
OUTPUT:
Ahora puedes ver esta publicación.
----------------------------------------------------------------------------------------------------

[4/16] Ejecutando BLOOMZ...
sample_id=sb_002 | model=bloomz_small | prompt=zero-shot | ruleset=R2
STATUS: success
TIME: 0.5966
OUTPUT:
Ahora puedes ver esta publicación en: https:/

In [43]:
runner_bloom.full_cleanup()
print("Limpieza BLOOMZ completa.")

Limpieza BLOOMZ completa.


In [44]:
bloom_log_path = Path("outputs/logs") / f"{experiment_id_bloom}.jsonl"

bloom_jsonl_rows = []
if bloom_log_path.exists():
    with open(bloom_log_path, "r", encoding="utf-8") as f:
        for line in f:
            bloom_jsonl_rows.append(json.loads(line))

bloom_logs_df = pd.DataFrame(bloom_jsonl_rows)

print("bloom_logs_df shape:", bloom_logs_df.shape)
bloom_logs_df.head(3)

bloom_logs_df shape: (16, 22)


,experiment_id,run_id,timestamp,dataset_name,fold_id,split_name,model_key,model_id,backend,prompt_type,...,generation_config,sample_id,source_text,reference_text,generated_text,prompt_text,inference_seconds,status,error_message,metrics
0,exp_small_batch_bloomz_20260308_145718,8952ead3-0aca-4fcc-a513-450354d5f30a,2026-03-08T14:57:18.357630,FEINA,None,None,bloomz_small,bigscience/bloomz-560m,transformers,zero-shot,...,"{'temperature': 0.2, 'top_p': 0.9, 'max_new_to...",sb_001,"Una vez dirimidos estos asuntos, se entra de l...",None,Ahora puedes leer este artículo.,Reescribe en español el siguiente texto con le...,6.6094,success,None,{}
1,exp_small_batch_bloomz_20260308_145718,d95e246b-fda0-4bd6-983b-cff98cde1419,2026-03-08T14:57:25.060636,FEINA,None,None,bloomz_small,bigscience/bloomz-560m,transformers,zero-shot,...,"{'temperature': 0.2, 'top_p': 0.9, 'max_new_to...",sb_001,"Una vez dirimidos estos asuntos, se entra de l...",None,Ahora puedes empezar tu propia empresa.,Reescribe en español el siguiente texto con le...,0.3283,success,None,{}
2,exp_small_batch_bloomz_20260308_145718,b0026ca4-5092-43da-b920-1aa11e3cf4b7,2026-03-08T14:57:25.484704,FEINA,None,None,bloomz_small,bigscience/bloomz-560m,transformers,zero-shot,...,"{'temperature': 0.2, 'top_p': 0.9, 'max_new_to...",sb_002,La implementación de estrategias de planificac...,None,Ahora puedes ver esta publicación.,Reescribe en español el siguiente texto con le...,0.2374,success,None,{}


In [45]:
bloom_logs_df["source_text"] = bloom_logs_df["source_text"].fillna("")
bloom_logs_df["generated_text"] = bloom_logs_df["generated_text"].fillna("")

bloom_logs_df["source_len_words"] = bloom_logs_df["source_text"].str.split().str.len()
bloom_logs_df["generated_len_words"] = bloom_logs_df["generated_text"].str.split().str.len()

bloom_logs_df["flag_english_artifact"] = bloom_logs_df["generated_text"].apply(has_english_artifact)
bloom_logs_df["flag_prompt_leak"] = bloom_logs_df["generated_text"].apply(has_prompt_leak)
bloom_logs_df["flag_weird_parenthesis"] = bloom_logs_df["generated_text"].apply(has_weird_parenthesis)
bloom_logs_df["flag_too_short"] = bloom_logs_df["generated_text"].apply(too_short_output)
bloom_logs_df["flag_too_long"] = bloom_logs_df.apply(
    lambda r: too_long_output(r["source_text"], r["generated_text"]), axis=1
)
bloom_logs_df["flag_telegraphic"] = bloom_logs_df["generated_text"].apply(too_telegraphic)
bloom_logs_df["flag_truncation"] = bloom_logs_df["generated_text"].apply(possible_truncation)

flag_cols = [
    "flag_english_artifact",
    "flag_prompt_leak",
    "flag_weird_parenthesis",
    "flag_too_short",
    "flag_too_long",
    "flag_telegraphic",
    "flag_truncation",
]

bloom_logs_df["n_flags"] = bloom_logs_df[flag_cols].sum(axis=1)

bloom_logs_df[[
    "sample_id", "model_key", "ruleset", "generated_text", "n_flags"
] + flag_cols]

,sample_id,model_key,ruleset,generated_text,n_flags,flag_english_artifact,flag_prompt_leak,flag_weird_parenthesis,flag_too_short,flag_too_long,flag_telegraphic,flag_truncation
0,sb_001,bloomz_small,R0,Ahora puedes leer este artículo.,0,False,False,False,False,False,False,False
1,sb_001,bloomz_small,R2,Ahora puedes empezar tu propia empresa.,0,False,False,False,False,False,False,False
2,sb_002,bloomz_small,R0,Ahora puedes ver esta publicación.,0,False,False,False,False,False,False,False
3,sb_002,bloomz_small,R2,Ahora puedes ver esta publicación en: https://...,1,False,False,False,False,False,True,False
4,sb_003,bloomz_small,R0,Ahora puedes leer este artículo.,0,False,False,False,False,False,False,False
5,sb_003,bloomz_small,R2,Ahora puedes ver las versiones anteriores del ...,0,False,False,False,False,False,False,False
6,sb_004,bloomz_small,R0,Ahora puedes leer esta publicación.,0,False,False,False,False,False,False,False
7,sb_004,bloomz_small,R2,Ahora puedes leer esta publicación.,0,False,False,False,False,False,False,False
8,sb_005,bloomz_small,R0,Ahora que hemos escrito todo lo anterior sobre...,0,False,False,False,False,False,False,False
9,sb_005,bloomz_small,R2,Ahora que hemos escrito todo lo anterior sobre...,0,False,False,False,False,False,False,False


In [46]:
combined_logs_df = pd.concat(
    [logs_df.copy(), bloom_logs_df.copy()],
    ignore_index=True
)

print("combined_logs_df shape:", combined_logs_df.shape)
combined_logs_df["model_key"].value_counts()

combined_logs_df shape: (48, 35)


model_key
mistral         16
llama3          16
bloomz_small    16
Name: count, dtype: int64

In [47]:
combined_logs_df["compression_ratio"] = (
    combined_logs_df["generated_len_words"] / combined_logs_df["source_len_words"]
)

final_summary_02 = (
    combined_logs_df.groupby(["model_key", "ruleset"])
    .agg(
        n_runs=("sample_id", "count"),
        success_rate=("status", lambda x: (x == "success").mean()),
        mean_time=("inference_seconds", "mean"),
        mean_flags=("n_flags", "mean"),
        mean_compression_ratio=("compression_ratio", "mean"),
    )
    .reset_index()
    .sort_values(["model_key", "ruleset"])
)

final_summary_02

,model_key,ruleset,n_runs,success_rate,mean_time,mean_flags,mean_compression_ratio
0,bloomz_small,R0,8,1.0,1.127525,0.125,0.326866
1,bloomz_small,R2,8,1.0,0.394688,0.125,0.389938
2,llama3,R0,8,1.0,7.886925,0.375,1.056872
3,llama3,R2,8,1.0,7.356162,0.125,0.985119
4,mistral,R0,8,1.0,6.648112,1.375,1.449677
5,mistral,R2,8,1.0,6.122425,1.000,1.067226
